# 39b. Calibration-sensitive common-frame training from 002.16 (one-window training)

Start from the **002.16 calibration-only checkpoint**. The previous 002.20/002.21 checkpoints learned substantial common-frame robustness but show near-zero rotation corrections on difficult rotated examples. Keep all physical streams, calibration targets, and sensor topology unchanged.

One 5-second window produces three related views: (1) original + calibration prior, (2) the same physical window and prior transformed by one shared body-frame SO(3), and (3) the same physical window with the **opposite calibration prior in the original frame**. No BPTT.

The objective is

\[L=L_{\mathrm{sup}}+0.75L_{\mathrm{active}}+0.05L_{\mathrm{eq}}+0.10L_{\mathrm{crosssigned}}.\]

The active supervised term emphasizes rotated-frame faults 2× relative to original-frame faults, normalized by total view weights. The cross-frame signed term compares **rotated positive correction minus transformed original-frame negative correction** against the exact difference of their targets; it penalizes zero corrections on active faults. Unlike the previous pair term, it directly constrains the rotated model output. It is rotation-only; translation and time-offset updates still receive ordinary supervision and equivariance consistency. Increasing the equivariance weight alone is *not* a remedy for zero predictions.

The source checkpoint, run directory, loss weights, checkpoint metadata, and diagnostics are separate from notebook 39. The previous checkpoint is not overwritten. All results below should be generated by executing this notebook on your local dataset; no saved results from the earlier run are reused.


## 1. Imports and repository root

In [27]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "obscalib").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the nn-obs-calib repository root containing src/obscalib.")

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from kaist_dataset.dataset import KAISTCalibrationDataset, KAISTRigCalibration, build_kaist_sequence_specs, make_kaist_dataloader
from obscalib.augmentations.config import AugmentationConfig, CalibrationEventConfig, CommonBodyFrameRandomizationConfig, FrameRandomizationConfig, NoiseAugmentationConfig, PerturbationMagnitudeConfig, PriorPerturbationConfig, SamplingRateAugmentationConfig, TrueCalibrationAugmentationConfig
from obscalib.augmentations.pipeline import AugmentationPipeline
from obscalib.augmentations import PerturbationDistribution
from obscalib.augmentations.prior_perturbation import CalibrationPriorPerturber, apply_prior_perturbation
from obscalib.calibration.state import CalibrationState
from obscalib.calibration.update import CalibrationUpdater
from obscalib.config import CalibrationPoolerConfig, MLPConfig, PooledCalibrationHeadConfig, PooledCalibrationModelConfig, TransformerConfig, WindowingConfig
from obscalib.data.collation import collate_windows
from obscalib.data.device import window_batch_to
from obscalib.data.structures import WindowBatch
from obscalib.geometry.lie import se3_log
from obscalib.geometry.processing import GeometryProcessor
from obscalib.models.pooled_calibration_model import PooledCalibrationModel
from obscalib.pipeline.window_step import WindowStep
from obscalib.tokenization.tokenizer import Tokenizer
from obscalib.training.checkpoint import load_training_checkpoint, save_training_checkpoint
from obscalib.training.metrics_logger import MetricsLogger
import gc

print("Repository root:", ROOT)
print("PyTorch:", torch.__version__)


Repository root: /home/camel/Skoltech/Mobile_Robotics_Lab/nn-obs-calib
PyTorch: 2.7.1+cu126


## 2. Independent-window refinement configuration


In [45]:
RUN_NAME = "kaist_pooled_calibration_paired_002.22_signed_crossframe_from_00216"

# Specify the exact previous successful checkpoint; this is a weights-only initialization.
# Change SOURCE_RUN_NAME to "kaist_pooled_calibration_paired_002.9_student" if that is your actual saved run directory.
SOURCE_RUN_NAME = "kaist_pooled_calibration_paired_002.16_correction_only"
RESUME_CHECKPOINT = ROOT / "runs" / SOURCE_RUN_NAME / "best.pt"

RANDOM_SEED = 12345
TRAIN_PAIR_SEED = 23456
VALIDATION_PAIR_SEED = 34567
TEST_PAIR_SEED = 45678
VISUALIZATION_PAIR_SEED = 56789

DEVICE_NAME = "cuda"
DEVICE = torch.device("cuda" if DEVICE_NAME == "auto" and torch.cuda.is_available() else DEVICE_NAME)
MODEL_DTYPE = torch.float32

CALIBRATION_KEYS = ("imu", "lidar", "lidar_right")
CALIBRATION_CONTEXT_DIM = 7

WINDOW_DURATION_S = 5.0
WINDOW_STRIDE_S = 5.0
MAX_SAMPLES_PER_SENSOR = 700

BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 1  # Effective optimizer batch 16; three-view graph fits memory more safely
NUM_WORKERS = 0
DROP_LAST_TRAIN = True

NUM_EPOCHS = 200
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0
CHECKPOINT_EVERY_EPOCHS = 3
PRINT_EVERY_BATCHES = 500
LR_REDUCTION_FACTOR = 0.5
LR_PATIENCE_EPOCHS = 8

MEASUREMENT_DIM = 6
TOKEN_INPUT_DIM = MEASUREMENT_DIM + 2
D_MODEL = 128
N_HEADS = 8
NUM_TRANSFORMER_LAYERS = 12
DIM_FEEDFORWARD = 512
TRANSFORMER_DROPOUT = 0.0
NUM_SUMMARY_TOKENS = 1

POOLER_N_HEADS = 8
POOLER_DROPOUT = 0.0
MLP_HIDDEN_DIMS = (256, 128)
SHARED_FEATURE_DIM = 128
MLP_DROPOUT = 0.0
HEAD_HIDDEN_DIMS = (128, 64)
HEAD_DROPOUT = 0.0

CORRECTION_LOSS_TYPE = "mse"
ROTATION_LOSS_SCALE_RAD = np.deg2rad(5.0)
TRANSLATION_LOSS_SCALE_M = 0.05
TIME_OFFSET_LOSS_SCALE_S = 0.050
ROTATION_LOSS_WEIGHT = 2.0
TRANSLATION_LOSS_WEIGHT = 1.0
TIME_OFFSET_LOSS_WEIGHT = 1.0
EQUIVARIANCE_LOSS_WEIGHT = 0.05  # Do NOT increase indiscriminately: zero outputs are already equivariant

# Extra gradient on the physically perturbed sensor. This is more direct than pair loss at countering near-zero corrections.
ACTIVE_ROTATION_SUPERVISION_WEIGHT = 0.75
ROTATED_ACTIVE_MULTIPLIER = 2.0  # Prioritize genuinely perturbed sensors in the rotated view

# Three instead of two forward passes when enabled; one extra *supervised* opposite-prior view.
# Set USE_SIGNED_PAIR=False for a two-view control, without changing the rest of the experiment.
USE_SIGNED_PAIR = True
OPPOSITE_VIEW_SUPERVISION_WEIGHT = 0.5  # Relative view weight; positive and rotated each have weight 1.
PAIR_LOSS_WEIGHT = 0.10 if USE_SIGNED_PAIR else 0.0  # Cross-frame signed ROTATION gap, not the old original-frame pair


# Fresh optimizer/scheduler from the strong 002.16 original-frame checkpoint; this is NOT an exact resume.
COSINE_MIN_LEARNING_RATE = 1e-6
RUN_FULL_TRAINING = True

CHANGE_EVENT_LOSS_WEIGHT = 0.0
CHANGE_TIME_LOSS_WEIGHT = 0.0

# One paired-prior distribution for every example; no fine/coarse branch.
PAIR_ROTATION_MAX_RAD = np.deg2rad(60.0)
PAIR_ROTATION_SCALE_RAD = np.deg2rad(30.0)
PAIR_TIME_OFFSET_MAX_S = 0.500
PAIR_TIME_OFFSET_SCALE_S = 0.250
STUDENT_T_DEGREES_OF_FREEDOM = 3
PAIR_TRANSLATION_MAX_M = 0.02  # Legacy disabled generic-prior config only.

# Physically consistent true-calibration augmentation, independent of the prior error.
TRUE_CALIBRATION_ROTATION_MAX_RAD = np.deg2rad(180.0)
TRUE_CALIBRATION_TIME_OFFSET_MAX_S = 0.030

# New shared-body-frame augmentation. Set to 0.0 for the control ablation.
COMMON_BODY_FRAME_PROBABILITY = 0.9  # Probability that the second view uses a nonidentity Haar SO(3) frame

if not RESUME_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Source checkpoint not found: {RESUME_CHECKPOINT}. Set SOURCE_RUN_NAME to the saved successful run.")

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Run:", RUN_NAME)
print("One-window frame-equivariance training (BPTT disabled); rotated-view probability:", COMMON_BODY_FRAME_PROBABILITY)
print("Initializing from:", RESUME_CHECKPOINT)
print("Calibration order:", CALIBRATION_KEYS)
print(f"Single prior: rotation maximum={np.rad2deg(PAIR_ROTATION_MAX_RAD):g}°, Student-t scale={np.rad2deg(PAIR_ROTATION_SCALE_RAD):g}°")
print(f"Single prior: tau maximum={PAIR_TIME_OFFSET_MAX_S * 1000:g} ms, Student-t scale={PAIR_TIME_OFFSET_SCALE_S * 1000:g} ms")
print(f"True calibration: rotation maximum={np.rad2deg(TRUE_CALIBRATION_ROTATION_MAX_RAD):g}°, tau maximum={TRUE_CALIBRATION_TIME_OFFSET_MAX_S * 1000:g} ms")

print("Equivariance weight:", EQUIVARIANCE_LOSS_WEIGHT, "rotated active multiplier:", ROTATED_ACTIVE_MULTIPLIER)

print("Signed opposite-prior branch:", USE_SIGNED_PAIR, "pair loss weight:", PAIR_LOSS_WEIGHT, "active emphasis:", ACTIVE_ROTATION_SUPERVISION_WEIGHT)


Run: kaist_pooled_calibration_paired_002.22_signed_crossframe_from_00216
One-window frame-equivariance training (BPTT disabled); rotated-view probability: 0.9
Initializing from: /home/camel/Skoltech/Mobile_Robotics_Lab/nn-obs-calib/runs/kaist_pooled_calibration_paired_002.16_correction_only/best.pt
Calibration order: ('imu', 'lidar', 'lidar_right')
Single prior: rotation maximum=60°, Student-t scale=30°
Single prior: tau maximum=500 ms, Student-t scale=250 ms
True calibration: rotation maximum=180°, tau maximum=30 ms
Equivariance weight: 0.05 rotated active multiplier: 2.0
Signed opposite-prior branch: True pair loss weight: 0.1 active emphasis: 0.75


## 3. KAIST splits and DataLoaders (unchanged from notebook 38)

In [46]:
KAIST_DATASET_ROOT = Path("/mnt/d/Downloads/MobRobLab/KAISTDataset")
LIDAR_POSE_ROOT = Path("/home/camel/Skoltech/phd_proposal/data/KAISTDataset")

TRAIN_NAMES = ["Urban08", "Urban12", "Urban13", "Urban14", "Urban15", "Urban16"]
VAL_NAMES = ["Urban26", "Urban28"]
TEST_NAMES = ["Urban38", "Urban39"]
ALL_SEQUENCE_NAMES = TRAIN_NAMES + VAL_NAMES + TEST_NAMES

if len(ALL_SEQUENCE_NAMES) != len(set(ALL_SEQUENCE_NAMES)):
    raise ValueError("Train, validation, and test sequence sets overlap.")

T_B_L = np.array([
    [-0.514066, -0.702201, -0.492595, -0.440699],
    [0.486485, -0.711672, 0.506809, 0.397052],
    [-0.706447, 0.0208933, 0.707457, 1.90953],
    [0.0, 0.0, 0.0, 1.0],
], dtype=float)

# Right LiDAR -> body, in the same convention as T_B_L for the left LiDAR.
T_B_L_right = np.array([
    [-0.512152,  0.699241, -0.498761, -0.449885],
    [-0.494811, -0.714859, -0.494104, -0.416713],
    [-0.702041, -0.0062642, 0.712109,  1.91294],
    [ 0.0,       0.0,       0.0,       1.0],
], dtype=float)

T_I_B = np.array([
    [1.0, 0.0, 0.0, -0.07],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.7],
    [0.0, 0.0, 0.0, 1.0],
], dtype=float)

rig_calibration = KAISTRigCalibration(T_B_I=np.linalg.inv(T_I_B), T_B_L=T_B_L, tau_imu_s=0.0, tau_lidar_s=0.0, T_B_L_right=T_B_L_right, tau_lidar_right_s=0.0)
windowing_config = WindowingConfig(window_duration_s=WINDOW_DURATION_S, window_stride_s=WINDOW_STRIDE_S, max_samples_per_sensor=MAX_SAMPLES_PER_SENSOR)

print("train:", TRAIN_NAMES)
print("validation:", VAL_NAMES)
print("test:", TEST_NAMES)


train: ['Urban08', 'Urban12', 'Urban13', 'Urban14', 'Urban15', 'Urban16']
validation: ['Urban26', 'Urban28']
test: ['Urban38', 'Urban39']


In [30]:
all_specs = build_kaist_sequence_specs(sequence_names=ALL_SEQUENCE_NAMES, kaist_dataset_root=KAIST_DATASET_ROOT, lidar_pose_root=LIDAR_POSE_ROOT, lidar_right_pose_filename="lidar_map_poses_vlp_right.csv")

# The 3-sensor experiment must never silently fall back to 2 sensors.
for spec in all_specs:
    if spec.lidar_right_pose_csv is None or not spec.lidar_right_pose_csv.is_file():
        raise FileNotFoundError(f"Missing required right-LiDAR pose CSV for {spec.name}: {spec.lidar_right_pose_csv}")
spec_by_name = {spec.name: spec for spec in all_specs}

train_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in TRAIN_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
val_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in VAL_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
test_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in TEST_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)

for split_name, dataset in (("train", train_dataset), ("validation", val_dataset), ("test", test_dataset)):
    if tuple(dataset.calibration_keys) != CALIBRATION_KEYS:
        raise AssertionError(f"{split_name} dataset has calibration keys {dataset.calibration_keys}, expected {CALIBRATION_KEYS}")

print(train_dataset.summary())
print()
print(val_dataset.summary())
print()
print(test_dataset.summary())


KAISTCalibrationDataset: 2006 windows from 6 sequences
approximate tensor storage: 44.0 MiB
  Urban08: windows=59, duration=297.8s, IMU=30800, LiDAR=2952, right LiDAR=2953
  Urban12: windows=473, duration=2478.6s, IMU=249198, LiDAR=23222, right LiDAR=23223
  Urban13: windows=223, duration=1446.5s, IMU=145492, LiDAR=10536, right LiDAR=10535
  Urban14: windows=286, duration=1737.0s, IMU=174681, LiDAR=13638, right LiDAR=13639
  Urban15: windows=184, duration=1034.3s, IMU=104795, LiDAR=9011, right LiDAR=9011
  Urban16: windows=781, duration=4458.1s, IMU=449215, LiDAR=37703, right LiDAR=37704

KAISTCalibrationDataset: 418 windows from 2 sequences
approximate tensor storage: 9.2 MiB
  Urban26: windows=94, duration=566.9s, IMU=58371, LiDAR=4553, right LiDAR=4551
  Urban28: windows=324, duration=1958.5s, IMU=197456, LiDAR=15651, right LiDAR=15658

KAISTCalibrationDataset: 699 windows from 2 sequences
approximate tensor storage: 15.4 MiB
  Urban38: windows=349, duration=2147.3s, IMU=216225, LiD

In [47]:
train_shuffle_generator = torch.Generator().manual_seed(RANDOM_SEED)
train_loader = make_kaist_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=DROP_LAST_TRAIN, num_workers=NUM_WORKERS, generator=train_shuffle_generator)
val_loader = make_kaist_dataloader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)
test_loader = make_kaist_dataloader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)

print("train batches:", len(train_loader))
print("validation batches:", len(val_loader))
print("test batches:", len(test_loader))


train batches: 250
validation batches: 53
test batches: 88


## 4. Physical augmentation plus **one** calibration prior

Keep true-calibration diversification, but do **not** apply common-body-frame augmentation inside this first stage. It will be applied once to the *already perturbed* window to create the second view, ensuring the physical calibration fault is the same in both coordinate systems. Independent sensor-frame randomization, sampling-rate augmentation, events, and noise retain notebook 38 settings (disabled here).

In [48]:
# Augment one physical installation, then sample one prior; apply common-frame G later to a complete view.
# The generic prior perturbation stays disabled to avoid modifying the shared truth.
context_augmentation_config = AugmentationConfig(
    sampling_rate=SamplingRateAugmentationConfig(enabled=False),
    frame_randomization=FrameRandomizationConfig(enabled=False),
    common_body_frame=CommonBodyFrameRandomizationConfig(enabled=False, probability=0.0),
    true_calibration=TrueCalibrationAugmentationConfig(
        enabled=True,
        probability=1.0,
        rotation_probability=1.0,
        translation_probability=0.0,
        time_offset_probability=1.0,
        rotation=PerturbationMagnitudeConfig(maximum=TRUE_CALIBRATION_ROTATION_MAX_RAD),
        time_offset=PerturbationMagnitudeConfig(maximum=TRUE_CALIBRATION_TIME_OFFSET_MAX_S),
    ),
    prior_perturbation=PriorPerturbationConfig(enabled=False, probability=1.0, rotation=PerturbationMagnitudeConfig(maximum=PAIR_ROTATION_MAX_RAD), translation=PerturbationMagnitudeConfig(maximum=PAIR_TRANSLATION_MAX_M), time_offset=PerturbationMagnitudeConfig(maximum=PAIR_TIME_OFFSET_MAX_S), one_calibration_per_sample=True, one_rotation_axis_per_sample=False),
    calibration_event=CalibrationEventConfig(enabled=False),
    noise=NoiseAugmentationConfig(enabled=False),
)

pair_prior_config = PriorPerturbationConfig(
    enabled=True,
    probability=1.0,
    rotation=PerturbationMagnitudeConfig(maximum=PAIR_ROTATION_MAX_RAD, distribution=PerturbationDistribution.TRUNCATED_STUDENT_T, scale=PAIR_ROTATION_SCALE_RAD, degrees_of_freedom=STUDENT_T_DEGREES_OF_FREEDOM),
    translation=None,
    time_offset=PerturbationMagnitudeConfig(maximum=PAIR_TIME_OFFSET_MAX_S, distribution=PerturbationDistribution.TRUNCATED_STUDENT_T, scale=PAIR_TIME_OFFSET_SCALE_S, degrees_of_freedom=STUDENT_T_DEGREES_OF_FREEDOM),
    one_calibration_per_sample=True,
    one_rotation_axis_per_sample=False,
    one_perturbation_type_per_sample=False,
)

context_augmentation_pipeline = AugmentationPipeline(context_augmentation_config)
pair_prior_perturber = CalibrationPriorPerturber(pair_prior_config)

print("Shared physical/context augmentation:", context_augmentation_config)
print("Paired prior:", pair_prior_config)
print("Common-frame stage disabled in physical pipeline:", context_augmentation_config.common_body_frame)
from obscalib.augmentations.common_body_frame import CommonBodyFrameRandomizer
frame_view_randomizer = CommonBodyFrameRandomizer(CommonBodyFrameRandomizationConfig(enabled=True, probability=COMMON_BODY_FRAME_PROBABILITY))
print("Paired-view randomizer:", frame_view_randomizer.config)


Shared physical/context augmentation: AugmentationConfig(sampling_rate=SamplingRateAugmentationConfig(enabled=False, minimum_imu_frequency_hz=None, minimum_lidar_frequency_hz=None, imu_probability=1.0, lidar_probability=1.0), frame_randomization=FrameRandomizationConfig(enabled=False), prior_perturbation=PriorPerturbationConfig(enabled=False, probability=1.0, probability_by_key={}, rotation=PerturbationMagnitudeConfig(maximum=1.0471975511965976, distribution=<PerturbationDistribution.UNIFORM: 'uniform'>, scale=None, degrees_of_freedom=3), translation=PerturbationMagnitudeConfig(maximum=0.02, distribution=<PerturbationDistribution.UNIFORM: 'uniform'>, scale=None, degrees_of_freedom=3), time_offset=PerturbationMagnitudeConfig(maximum=0.5, distribution=<PerturbationDistribution.UNIFORM: 'uniform'>, scale=None, degrees_of_freedom=3), one_calibration_per_sample=True, one_rotation_axis_per_sample=False, one_perturbation_type_per_sample=False), calibration_event=CalibrationEventConfig(enabled

## 5. Pooled model and complete window pipeline (architecture unchanged)

In [49]:
model_config = PooledCalibrationModelConfig(
    transformer=TransformerConfig(input_dim=TOKEN_INPUT_DIM, d_model=D_MODEL, n_heads=N_HEADS, num_layers=NUM_TRANSFORMER_LAYERS, dim_feedforward=DIM_FEEDFORWARD, dropout=TRANSFORMER_DROPOUT, num_summary_tokens=NUM_SUMMARY_TOKENS, activation="gelu"),
    mlp=MLPConfig(hidden_dims=MLP_HIDDEN_DIMS, output_dim=SHARED_FEATURE_DIM, activation="gelu", dropout=MLP_DROPOUT),
    pooler=CalibrationPoolerConfig(
        n_heads=POOLER_N_HEADS,
        dropout=POOLER_DROPOUT,
        calibration_context_dim=0,           # CALIBRATION_CONTEXT_DIM
    ),

    head=PooledCalibrationHeadConfig(
        calibration_context_dim=0,
        hidden_dims=HEAD_HIDDEN_DIMS,
        activation="gelu",
        dropout=HEAD_DROPOUT,
    ),
)

geometry_processor = GeometryProcessor()
tokenizer = Tokenizer(measurement_dim=MEASUREMENT_DIM, calibration_keys=CALIBRATION_KEYS)
model = PooledCalibrationModel(model_config).to(device=DEVICE, dtype=MODEL_DTYPE)
calibration_updater = CalibrationUpdater()
window_step = WindowStep(geometry_processor=geometry_processor, tokenizer=tokenizer, model=model, calibration_updater=calibration_updater)

# if not getattr(model, "uses_calibration_context", False):
#     raise RuntimeError("This experiment requires the pooled model to use numeric calibration context.")

print(model)
print()
print("Parameters:", sum(parameter.numel() for parameter in model.parameters()))
print("Trainable parameters:", sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))
print("Pooler calibration context dim:", model.pooler.calibration_context_dim)
print("Head calibration context dim:", model.head.config.calibration_context_dim)


PooledCalibrationModel(
  (transformer_encoder): SummaryTokenTransformerEncoder(
    (input_projection): Linear(in_features=8, out_features=128, bias=True)
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-11): 12 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=512, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (linear2): Linear(in_features=512, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.0, inplace=False)
          (dropout2): Dropout(p=0.0, inplace=False)
        )
      )
    )
  )
  (pooler): CalibrationPooler(
    (attention): MultiheadAttention(
      (out_proj): NonDynamicallyQ

## 6. Construct physically matched original, rotated, and opposite-prior views

The opposite view differs only in the sign of the *one sampled prior perturbation*. It shares the physical measurements, true calibration, noise realization and sampling pattern with the other views. All targets are recomputed from the prior and the same true calibration. No independent physical simulation is performed for the opposite branch.


In [50]:
def make_augmentation_generator(seed: int):
    return torch.Generator(device=DEVICE.type).manual_seed(seed)


def clone_window_with_calibration(window: WindowBatch, calibration):
    return WindowBatch(streams=dict(window.streams), current_calibration=calibration, metadata=dict(window.metadata), targets=None if window.targets is None else dict(window.targets), dataset_indices=window.dataset_indices)



def make_equivariant_windows(base_window: WindowBatch, *, generator: torch.Generator, frame_probability: float | None = None, return_negative: bool = False):
    """
    Construct two or three views of the same physical sensor measurements.

    View 1: original_window
        Raw measurements with an erroneous calibration prior T_plus,
        expressed in the original common body frame B.

    View 2: rotated_window
        Exactly the same raw measurements and the same physical calibration error,
        but all sensor extrinsics and calibration targets are expressed in a new
        common body frame B' obtained by applying H.

    View 3: opposite_window (optional)
        Exactly the same raw measurements and true calibration as View 1,
        but with the opposite prior perturbation T_minus, still expressed in B.

    Returns:
        original_window: WindowBatch containing the positive prior in frame B.
        rotated_window: WindowBatch containing the same prior in frame B'.
        H:              [batch_size, 4, 4] shared SE(3) frame transformation.
        active:         [batch_size] boolean mask for nonidentity frame rotations.
        opposite_window: Optional WindowBatch containing the negative prior in B.
    """

    context_result = context_augmentation_pipeline(base_window, generator=generator)

    physical_window = context_result.augmented_window

    # These are the TRUE calibration states after physical augmentation.
    true_calibration = physical_window.current_calibration

    # Sample one calibration-prior perturbation for each batch example.
    _, delta_xi, delta_tau = pair_prior_perturber(true_calibration, generator=generator)

    # Apply the sampled perturbations to the true calibration.
    candidate_calibration = apply_prior_perturbation(true_calibration, delta_xi, delta_tau)

    # VIEW 1: ORIGINAL
    # Keep the physical sensor streams unchanged, but replace their
    # calibration estimates with the erroneous candidate calibration.
    original_window = clone_window_with_calibration(physical_window, candidate_calibration)

    # Choose the shared-body-frame augmenter.
    augmenter = frame_view_randomizer if frame_probability is None else CommonBodyFrameRandomizer(CommonBodyFrameRandomizationConfig(enabled=True, probability=frame_probability))

    # VIEW 2: ROTATED COMMON FRAME
    # Sample ONE rotation G per batch example and apply it to EVERY sensor.
    rotated_window, _, H = augmenter(original_window, generator=generator)

    # Detect which batch examples actually received a nonidentity rotation.
    active = torch.linalg.vector_norm(H[:, :3, :3] - torch.eye(3, dtype=H.dtype, device=H.device), dim=(1, 2)) > 1e-5

    if not return_negative:
        return original_window, rotated_window, H, active

    # Construct the OPPOSITE calibration perturbation by negating
    # the originally sampled SE(3) and time-offset perturbation vectors.
    opposite_calibration = apply_prior_perturbation(true_calibration, {key: -value for key, value in delta_xi.items()}, {key: -value for key, value in delta_tau.items()})

    # VIEW 3: OPPOSITE PRIOR
    # Keep the SAME physical measurements and original body frame B,
    # but replace T_plus with T_minus.
    opposite_window = clone_window_with_calibration(physical_window, opposite_calibration)

    return original_window, rotated_window, H, active, opposite_window


def correction_target(window: WindowBatch, calibration_key: str):
    if window.targets is None or calibration_key not in window.targets:
        raise ValueError(f"Missing calibration target for {calibration_key!r}.")
    target = window.targets[calibration_key]
    current = window.current_calibration[calibration_key]
    if target.next_transform is None or target.next_time_offset is None:
        raise ValueError(f"Incomplete transform/clock target for {calibration_key!r}.")
    return se3_log(target.next_transform @ torch.linalg.inv(current.transform)), target.next_time_offset - current.time_offset


def rotate_twist(delta_xi: torch.Tensor, G: torch.Tensor) -> torch.Tensor:
    """Adjoint action for H=[G,0;0,1] and twist ordering [rotation, translation]."""
    return torch.cat((torch.bmm(G, delta_xi[:, :3, None]).squeeze(-1), torch.bmm(G, delta_xi[:, 3:, None]).squeeze(-1)), dim=-1)


## 7. Geometric and supervised-target semantic checks

These checks verify the transformed view preserves raw IMU/LiDAR data, timestamps, and relative extrinsics, and that **targets actually obey the expected adjoint transformation**. They are prerequisites to trusting the equivariance loss. The common-frame pair is intentionally built *after* the prior has been sampled.

In [51]:
semantic_base_window = window_batch_to(next(iter(train_loader)), DEVICE, dtype=MODEL_DTYPE)
semantic_original, semantic_rotated, semantic_H, semantic_active = make_equivariant_windows(semantic_base_window, generator=make_augmentation_generator(TRAIN_PAIR_SEED + 1), frame_probability=1.0)
G = semantic_H[:, :3, :3]
assert bool(semantic_active.all()), "The semantic check must rotate every example."
for stream_key in semantic_original.streams:
    a, b = semantic_original.streams[stream_key], semantic_rotated.streams[stream_key]
    torch.testing.assert_close(a.values, b.values, rtol=0, atol=0)
    torch.testing.assert_close(a.timestamps, b.timestamps, rtol=0, atol=0)
    if a.interval_start_timestamps is not None:
        torch.testing.assert_close(a.interval_start_timestamps, b.interval_start_timestamps, rtol=0, atol=0)
for key in CALIBRATION_KEYS:
    a, b = semantic_original.current_calibration[key], semantic_rotated.current_calibration[key]
    torch.testing.assert_close(b.transform, semantic_H @ a.transform, rtol=1e-5, atol=1e-5)
    torch.testing.assert_close(b.time_offset, a.time_offset, rtol=0, atol=0)
    orig_xi, orig_tau = correction_target(semantic_original, key)
    rot_xi, rot_tau = correction_target(semantic_rotated, key)
    torch.testing.assert_close(rot_xi, rotate_twist(orig_xi, G), rtol=5e-4, atol=5e-5)
    torch.testing.assert_close(rot_tau, orig_tau, rtol=0, atol=1e-7)
    for other in CALIBRATION_KEYS:
        rel_original = torch.linalg.inv(a.transform) @ semantic_original.current_calibration[other].transform
        rel_rotated = torch.linalg.inv(b.transform) @ semantic_rotated.current_calibration[other].transform
        torch.testing.assert_close(rel_original, rel_rotated, rtol=1e-5, atol=1e-5)
print("PASS: shared frame, identical raw sensor measurements, fixed relative extrinsics, adjoint-consistent supervision.")
print("Median paired frame rotation [deg]:", torch.rad2deg(torch.linalg.vector_norm(se3_log(semantic_H)[:, :3], dim=-1)).median().item())

# Check exact opposite-prior correction targets, including the time offset, without assuming small angles.
semantic_plus, _, _, _, semantic_minus = make_equivariant_windows(window_batch_to(next(iter(train_loader)), DEVICE, dtype=MODEL_DTYPE), generator=make_augmentation_generator(TRAIN_PAIR_SEED + 2), frame_probability=1.0, return_negative=True)
for key in CALIBRATION_KEYS:
    xi_plus, tau_plus = correction_target(semantic_plus, key)
    xi_minus, tau_minus = correction_target(semantic_minus, key)
    torch.testing.assert_close(xi_plus, -xi_minus, rtol=1e-3, atol=2e-4)
    torch.testing.assert_close(tau_plus, -tau_minus, rtol=1e-4, atol=2e-6)
    for stream_key in semantic_plus.streams:
        torch.testing.assert_close(semantic_plus.streams[stream_key].values, semantic_minus.streams[stream_key].values, rtol=0, atol=0)
print("PASS: positive/negative prior targets are opposite and share the physical measurements.")
del semantic_plus, semantic_minus

del semantic_base_window, semantic_original, semantic_rotated, semantic_H, semantic_active, G
if DEVICE.type == "cuda": torch.cuda.empty_cache()


PASS: shared frame, identical raw sensor measurements, fixed relative extrinsics, adjoint-consistent supervision.
Median paired frame rotation [deg]: 85.0155029296875
PASS: positive/negative prior targets are opposite and share the physical measurements.


## 8. Supervised correction, active-fault emphasis, equivariance, and optional pair-difference objective

**Why the cross-frame signed pair differs from merely forcing opposite predictions:** `L_pair` compares predicted signed **differences** with differences of the exact supervised correction targets. Zero predictions on an active fault therefore incur a positive pair loss. This information is algebraically implied by the two supervised targets; it is a regularizer/gradient reweighting, not new LiDAR or IMU information.

The additional active-rotation term is the primary intervention for the observed line `rotated error ≈ target magnitude`, because `one_calibration_per_sample=True` leaves two calibration heads with zero targets. A zero-output baseline must remain visible in the held-out metrics.


**39b changes:** active rotation supervision receives coefficient 0.75, with 2× emphasis on rotated-frame active faults. The `pair` term is now the **cross-frame signed ROTATION gap** (0.10), using rotated-positive and transformed original-negative predictions. Equivariance weight remains 0.05 to avoid rewarding two jointly shrunken predictions. The `pair` metric no longer includes temporal or translational differences; those remain supervised by the regular losses.


In [52]:
def correction_regression_loss(prediction, target):
    if CORRECTION_LOSS_TYPE == "mse": return F.mse_loss(prediction, target)
    if CORRECTION_LOSS_TYPE == "smooth_l1": return F.smooth_l1_loss(prediction, target, beta=1.0)
    if CORRECTION_LOSS_TYPE == "l1": return F.l1_loss(prediction, target)
    raise ValueError(f"Unknown CORRECTION_LOSS_TYPE={CORRECTION_LOSS_TYPE!r}.")


def compute_training_loss(window, output):
    total = torch.zeros((), dtype=MODEL_DTYPE, device=DEVICE)
    metrics = {}
    active_rotation_errors = []
    active_tau_errors = []
    for key in CALIBRATION_KEYS:
        pred = output.predictions[key]
        target_xi, target_tau = correction_target(window, key)
        lr = correction_regression_loss(pred.delta_xi[:, :3] / ROTATION_LOSS_SCALE_RAD, target_xi[:, :3] / ROTATION_LOSS_SCALE_RAD)
        lt = correction_regression_loss(pred.delta_xi[:, 3:] / TRANSLATION_LOSS_SCALE_M, target_xi[:, 3:] / TRANSLATION_LOSS_SCALE_M)
        l_tau = correction_regression_loss(pred.delta_tau / TIME_OFFSET_LOSS_SCALE_S, target_tau / TIME_OFFSET_LOSS_SCALE_S)
        total = total + ROTATION_LOSS_WEIGHT * lr + TRANSLATION_LOSS_WEIGHT * lt + TIME_OFFSET_LOSS_WEIGHT * l_tau
        metrics[f"{key}/rotation_error_deg"] = torch.rad2deg(torch.linalg.vector_norm(pred.delta_xi[:, :3] - target_xi[:, :3], dim=-1)).mean().item()
        metrics[f"{key}/time_offset_error_ms"] = (1000.0 * torch.abs(pred.delta_tau - target_tau)).mean().item()
        metrics[f"{key}/translation_error_cm"] = (100.0 * torch.linalg.vector_norm(pred.delta_xi[:, 3:] - target_xi[:, 3:], dim=-1)).mean().item()
        rotation_active = torch.linalg.vector_norm(target_xi[:, :3], dim=-1) > 1e-6
        tau_active = target_tau[:, 0].abs() > 1e-7
        metrics[f"{key}/rotation_active_fraction"] = rotation_active.float().mean().item()
        metrics[f"{key}/tau_active_fraction"] = tau_active.float().mean().item()
        if bool(rotation_active.any()): active_rotation_errors.append(torch.rad2deg(torch.linalg.vector_norm(pred.delta_xi[rotation_active, :3] - target_xi[rotation_active, :3], dim=-1)))
        if bool(tau_active.any()): active_tau_errors.append(1000.0 * (pred.delta_tau[tau_active] - target_tau[tau_active]).abs().reshape(-1))
    total = total / len(CALIBRATION_KEYS)
    # These are evaluated ONLY where the calibration target really requires a correction.
    # They distinguish learning the injected errors from merely predicting zero for healthy sensors.
    metrics["active_rotation_error_deg"] = torch.cat(active_rotation_errors).mean().item() if active_rotation_errors else 0.0
    metrics["active_tau_error_ms"] = torch.cat(active_tau_errors).mean().item() if active_tau_errors else 0.0
    metrics["active_rotation_count"] = sum(len(x) for x in active_rotation_errors)
    metrics["active_tau_count"] = sum(len(x) for x in active_tau_errors)
    return total, metrics


def compute_equivariance_loss(original_output, rotated_output, G, active):
    losses = []
    metrics = {"active_fraction": active.float().mean().item()}
    for key in CALIBRATION_KEYS:
        pa = original_output.predictions[key]
        pb = rotated_output.predictions[key]
        expected_xi = rotate_twist(pa.delta_xi, G)
        if bool(active.any()):
            r_err = (pb.delta_xi[active, :3] - expected_xi[active, :3]) / ROTATION_LOSS_SCALE_RAD
            t_err = (pb.delta_xi[active, 3:] - expected_xi[active, 3:]) / TRANSLATION_LOSS_SCALE_M
            tau_err = (pb.delta_tau[active] - pa.delta_tau[active]) / TIME_OFFSET_LOSS_SCALE_S
            rotation_loss = r_err.square().mean()
            translation_loss = t_err.square().mean()
            time_loss = tau_err.square().mean()
            metrics[f"{key}/rotation_equivariance_deg"] = torch.rad2deg(torch.linalg.vector_norm(r_err * ROTATION_LOSS_SCALE_RAD, dim=-1)).mean().item()
            metrics[f"{key}/tau_equivariance_ms"] = (1000.0 * tau_err.abs() * TIME_OFFSET_LOSS_SCALE_S).mean().item()
        else:
            zero = (pa.delta_xi.sum() + pb.delta_xi.sum() + pa.delta_tau.sum() + pb.delta_tau.sum()) * 0.0
            rotation_loss = translation_loss = time_loss = zero
            metrics[f"{key}/rotation_equivariance_deg"] = 0.0
            metrics[f"{key}/tau_equivariance_ms"] = 0.0
        losses.append(ROTATION_LOSS_WEIGHT * rotation_loss + TRANSLATION_LOSS_WEIGHT * translation_loss + TIME_OFFSET_LOSS_WEIGHT * time_loss)
        metrics[f"{key}/loss_rotation"] = rotation_loss.detach().item()
        metrics[f"{key}/loss_translation"] = translation_loss.detach().item()
        metrics[f"{key}/loss_time_offset"] = time_loss.detach().item()
    return torch.stack(losses).mean(), metrics



def compute_active_rotation_supervision(window, output):
    """Additional MSE only on *injected* rotation targets, aggregated over active keys and examples."""
    discrepancies = []
    predicted_norms = []
    target_norms = []
    for key in CALIBRATION_KEYS:
        target_xi, _ = correction_target(window, key)
        target_phi = target_xi[:, :3]
        pred_phi = output.predictions[key].delta_xi[:, :3]
        mask = torch.linalg.vector_norm(target_phi, dim=-1) > 1e-6
        if bool(mask.any()):
            discrepancies.append(((pred_phi[mask] - target_phi[mask]) / ROTATION_LOSS_SCALE_RAD).square().mean(dim=-1))
            predicted_norms.append(torch.rad2deg(torch.linalg.vector_norm(pred_phi[mask], dim=-1)))
            target_norms.append(torch.rad2deg(torch.linalg.vector_norm(target_phi[mask], dim=-1)))
    if not discrepancies:
        first = output.predictions[CALIBRATION_KEYS[0]].delta_xi
        return first.sum() * 0.0, {"active_rotation_target_deg": 0.0, "active_rotation_prediction_deg": 0.0}
    rotation_loss = ROTATION_LOSS_WEIGHT * torch.cat(discrepancies).mean()
    return rotation_loss, {"active_rotation_target_deg": torch.cat(target_norms).mean().item(), "active_rotation_prediction_deg": torch.cat(predicted_norms).mean().item()}


def compute_signed_pair_difference_loss(positive_window, positive_output, negative_window, negative_output):
    """Supervised difference loss for the same physical samples under +/- errors; only selected fault components contribute."""
    rotation_errors, translation_errors, time_errors = [], [], []
    target_differences, predicted_differences = [], []
    for key in CALIBRATION_KEYS:
        pos_target_xi, pos_target_tau = correction_target(positive_window, key)
        neg_target_xi, neg_target_tau = correction_target(negative_window, key)
        pos_prediction = positive_output.predictions[key]
        neg_prediction = negative_output.predictions[key]
        target_diff_xi = pos_target_xi - neg_target_xi
        predicted_diff_xi = pos_prediction.delta_xi - neg_prediction.delta_xi
        target_diff_tau = pos_target_tau - neg_target_tau
        predicted_diff_tau = pos_prediction.delta_tau - neg_prediction.delta_tau
        rotation_mask = torch.linalg.vector_norm(target_diff_xi[:, :3], dim=-1) > 1e-6
        translation_mask = torch.linalg.vector_norm(target_diff_xi[:, 3:], dim=-1) > 1e-6
        time_mask = target_diff_tau[:, 0].abs() > 1e-7
        if bool(rotation_mask.any()):
            rotation_errors.append(((predicted_diff_xi[rotation_mask, :3] - target_diff_xi[rotation_mask, :3]) / ROTATION_LOSS_SCALE_RAD).square().mean(dim=-1))
            target_differences.append(torch.linalg.vector_norm(target_diff_xi[rotation_mask, :3], dim=-1))
            predicted_differences.append(torch.linalg.vector_norm(predicted_diff_xi[rotation_mask, :3], dim=-1))
        if bool(translation_mask.any()):
            translation_errors.append(((predicted_diff_xi[translation_mask, 3:] - target_diff_xi[translation_mask, 3:]) / TRANSLATION_LOSS_SCALE_M).square().mean(dim=-1))
        if bool(time_mask.any()):
            time_errors.append(((predicted_diff_tau[time_mask, 0] - target_diff_tau[time_mask, 0]) / TIME_OFFSET_LOSS_SCALE_S).square())
    zero = positive_output.predictions[CALIBRATION_KEYS[0]].delta_xi.sum() * 0.0
    lr = torch.cat(rotation_errors).mean() if rotation_errors else zero
    lt = torch.cat(translation_errors).mean() if translation_errors else zero
    l_tau = torch.cat(time_errors).mean() if time_errors else zero
    pair_loss = ROTATION_LOSS_WEIGHT * lr + TRANSLATION_LOSS_WEIGHT * lt + TIME_OFFSET_LOSS_WEIGHT * l_tau
    metrics = {"loss_rotation": lr.detach().item(), "loss_translation": lt.detach().item(), "loss_time_offset": l_tau.detach().item(), "active_rotation_pairs": sum(x.numel() for x in target_differences)}
    if target_differences:
        metrics["target_rotation_gap_deg"] = torch.rad2deg(torch.cat(target_differences)).mean().item()
        metrics["predicted_rotation_gap_deg"] = torch.rad2deg(torch.cat(predicted_differences)).mean().item()
    else:
        metrics["target_rotation_gap_deg"] = metrics["predicted_rotation_gap_deg"] = 0.0
    return pair_loss, metrics


def compute_cross_frame_signed_rotation_loss(rotated_window, rotated_output, opposite_window, opposite_output, G):
    """Learn the signed calibration response *in the rotated frame*, without a fourth forward pass.

    The opposite-prior prediction (original coordinates) and its exact target are
    both transformed with G. The difference must match the actual difference
    between the two correction targets. All-zero predictions incur a positive
    penalty whenever a nonzero rotation fault is present.
    """
    rotation_errors, predicted_gaps, true_gaps = [], [], []
    for key in CALIBRATION_KEYS:
        rotated_xi, _ = correction_target(rotated_window, key)
        opposite_xi, _ = correction_target(opposite_window, key)
        true_gap = rotated_xi[:, :3] - torch.bmm(G, opposite_xi[:, :3, None]).squeeze(-1)
        pred_rotated = rotated_output.predictions[key].delta_xi[:, :3]
        pred_opposite = opposite_output.predictions[key].delta_xi[:, :3]
        predicted_gap = pred_rotated - torch.bmm(G, pred_opposite[:, :, None]).squeeze(-1)
        fault_mask = torch.linalg.vector_norm(true_gap, dim=-1) > 1e-6
        if bool(fault_mask.any()):
            rotation_errors.append(((predicted_gap[fault_mask] - true_gap[fault_mask]) / ROTATION_LOSS_SCALE_RAD).square().mean(dim=-1))
            predicted_gaps.append(torch.rad2deg(torch.linalg.vector_norm(predicted_gap[fault_mask], dim=-1)))
            true_gaps.append(torch.rad2deg(torch.linalg.vector_norm(true_gap[fault_mask], dim=-1)))
    if not rotation_errors:
        zero = rotated_output.predictions[CALIBRATION_KEYS[0]].delta_xi.sum() * 0.0
        return zero, {"target_rotation_gap_deg": 0.0, "predicted_rotation_gap_deg": 0.0, "active_rotation_pairs": 0}
    loss = ROTATION_LOSS_WEIGHT * torch.cat(rotation_errors).mean()
    return loss, {
        "target_rotation_gap_deg": torch.cat(true_gaps).mean().item(),
        "predicted_rotation_gap_deg": torch.cat(predicted_gaps).mean().item(),
        "active_rotation_pairs": sum(x.numel() for x in true_gaps),
    }


def compute_equivariant_objective(original_window, original_output, rotated_window, rotated_output, common_frame, active, opposite_window=None, opposite_output=None):
    if (opposite_window is None) != (opposite_output is None):
        raise ValueError("opposite_window and opposite_output must both be supplied or both be omitted.")
    original_loss, original_metrics = compute_training_loss(original_window, original_output)
    rotated_loss, rotated_metrics = compute_training_loss(rotated_window, rotated_output)
    original_active_loss, original_active_metrics = compute_active_rotation_supervision(original_window, original_output)
    rotated_active_loss, rotated_active_metrics = compute_active_rotation_supervision(rotated_window, rotated_output)
    equiv_loss, equiv_metrics = compute_equivariance_loss(original_output, rotated_output, common_frame[:, :3, :3], active)
    view_weight = 2.0
    supervision_sum = original_loss + rotated_loss
    active_sum = original_active_loss + ROTATED_ACTIVE_MULTIPLIER * rotated_active_loss
    active_view_weight = 1.0 + ROTATED_ACTIVE_MULTIPLIER
    pair_loss = original_loss * 0.0
    pair_metrics = {}
    opposite_metrics = {}
    if opposite_window is not None:
        opposite_loss, opposite_metrics = compute_training_loss(opposite_window, opposite_output)
        opposite_active_loss, opposite_active_metrics = compute_active_rotation_supervision(opposite_window, opposite_output)
        supervision_sum = supervision_sum + OPPOSITE_VIEW_SUPERVISION_WEIGHT * opposite_loss
        active_sum = active_sum + OPPOSITE_VIEW_SUPERVISION_WEIGHT * opposite_active_loss
        active_view_weight += OPPOSITE_VIEW_SUPERVISION_WEIGHT
        view_weight += OPPOSITE_VIEW_SUPERVISION_WEIGHT
        pair_loss, pair_metrics = compute_cross_frame_signed_rotation_loss(rotated_window, rotated_output, opposite_window, opposite_output, common_frame[:, :3, :3])
        opposite_metrics.update(opposite_active_metrics)
    elif PAIR_LOSS_WEIGHT > 0.0:
        raise ValueError("Nonzero PAIR_LOSS_WEIGHT requires the opposite-prior view.")
    supervised = supervision_sum / view_weight
    active_supervised = active_sum / active_view_weight
    supervised_selection = supervised + ACTIVE_ROTATION_SUPERVISION_WEIGHT * active_supervised
    loss = supervised_selection + EQUIVARIANCE_LOSS_WEIGHT * equiv_loss + PAIR_LOSS_WEIGHT * pair_loss
    metrics = {
        "loss_total": loss.detach().item(), "loss_regular": 0.5 * (original_loss + rotated_loss).detach().item(),
        "loss_supervised": supervised.detach().item(), "loss_active_supervised": active_supervised.detach().item(),
        "loss_supervised_selection": supervised_selection.detach().item(), "loss_equivariance": equiv_loss.detach().item(),
        "loss_pair": pair_loss.detach().item(), "loss_original": original_loss.detach().item(), "loss_rotated": rotated_loss.detach().item(),
        "weighted_active": ACTIVE_ROTATION_SUPERVISION_WEIGHT * active_supervised.detach().item(),
        "weighted_equivariance": EQUIVARIANCE_LOSS_WEIGHT * equiv_loss.detach().item(),
        "weighted_pair": PAIR_LOSS_WEIGHT * pair_loss.detach().item(),
    }
    metrics.update({f"original/{k}": v for k, v in original_metrics.items()})
    metrics.update({f"rotated/{k}": v for k, v in rotated_metrics.items()})
    metrics.update({f"equiv/{k}": v for k, v in equiv_metrics.items()})
    metrics.update({f"pair/{k}": v for k, v in pair_metrics.items()})
    metrics.update({f"opposite/{k}": v for k, v in opposite_metrics.items()})
    metrics.update({f"original/{k}": v for k, v in original_active_metrics.items()})
    metrics.update({f"rotated/{k}": v for k, v in rotated_active_metrics.items()})
    return loss, metrics


optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=COSINE_MIN_LEARNING_RATE)


## 9. Three-view gradient and oracle checks

The tests assert that exact correction targets make the signed-pair loss vanish, while all-zero predictions incur a nonzero penalty for an active fault. They also check that the implemented full training objective backpropagates. These tests do not substitute for running the actual dataset and model locally.


In [53]:
from types import SimpleNamespace

smoke_base = window_batch_to(collate_windows([train_dataset[i] for i in (0, min(len(train_dataset) - 1, 100))]), DEVICE, dtype=MODEL_DTYPE)
smoke_original, smoke_rotated, smoke_H, smoke_active, smoke_opposite = make_equivariant_windows(smoke_base, generator=make_augmentation_generator(TRAIN_PAIR_SEED + 22), frame_probability=1.0, return_negative=True)

# Algebraic oracle checks for the opposite-signed *target differences*, independent of network initialization.
def _dummy_calibration_output(window, *, predict_zero: bool):
    predictions = {}
    for key in CALIBRATION_KEYS:
        true_xi, true_tau = correction_target(window, key)
        predictions[key] = SimpleNamespace(delta_xi=torch.zeros_like(true_xi) if predict_zero else true_xi, delta_tau=torch.zeros_like(true_tau) if predict_zero else true_tau)
    return SimpleNamespace(predictions=predictions)

oracle_pair, _ = compute_signed_pair_difference_loss(smoke_original, _dummy_calibration_output(smoke_original, predict_zero=False), smoke_opposite, _dummy_calibration_output(smoke_opposite, predict_zero=False))
zero_pair, _ = compute_signed_pair_difference_loss(smoke_original, _dummy_calibration_output(smoke_original, predict_zero=True), smoke_opposite, _dummy_calibration_output(smoke_opposite, predict_zero=True))
assert abs(oracle_pair.item()) < 1e-8, f"Oracle pair loss should be zero: {oracle_pair.item()}"
assert zero_pair.item() > 0.1, "All-zero outputs must NOT solve supervised signed-pair loss."
print(f"Pair oracle: {oracle_pair.item():.7f} | all-zero baseline: {zero_pair.item():.3f}")
oracle_cross, _ = compute_cross_frame_signed_rotation_loss(smoke_rotated, _dummy_calibration_output(smoke_rotated, predict_zero=False), smoke_opposite, _dummy_calibration_output(smoke_opposite, predict_zero=False), smoke_H[:, :3, :3])
zero_cross, _ = compute_cross_frame_signed_rotation_loss(smoke_rotated, _dummy_calibration_output(smoke_rotated, predict_zero=True), smoke_opposite, _dummy_calibration_output(smoke_opposite, predict_zero=True), smoke_H[:, :3, :3])
assert abs(oracle_cross.item()) < 1e-6, f"Cross-frame oracle loss should be zero: {oracle_cross.item()}"
assert zero_cross.item() > 0.1, "All-zero outputs must NOT solve the cross-frame signed rotation loss."
print(f"Cross-frame signed oracle: {oracle_cross.item():.7f} | all-zero penalty: {zero_cross.item():.3f}")

model.train()
optimizer.zero_grad(set_to_none=True)
smoke_original_out = window_step(smoke_original).model_output
smoke_rotated_out = window_step(smoke_rotated).model_output
smoke_opposite_out = window_step(smoke_opposite).model_output if USE_SIGNED_PAIR else None
smoke_loss, smoke_metrics = compute_equivariant_objective(smoke_original, smoke_original_out, smoke_rotated, smoke_rotated_out, smoke_H, smoke_active, smoke_opposite if USE_SIGNED_PAIR else None, smoke_opposite_out)
smoke_loss.backward()
grad_norm = sum(float(p.grad.detach().square().sum().cpu()) for p in model.parameters() if p.grad is not None) ** 0.5
print("Smoke objective:", {k: round(v, 4) for k, v in smoke_metrics.items() if k.startswith('loss_') or k.startswith('weighted_')}, "gradient norm:", grad_norm)
assert np.isfinite(smoke_loss.item()) and np.isfinite(grad_norm) and grad_norm > 0
assert smoke_metrics["equiv/active_fraction"] > 0.99
print("THREE-VIEW GRADIENT SMOKE: PASS")
optimizer.zero_grad(set_to_none=True)
del smoke_base, smoke_original, smoke_rotated, smoke_opposite, smoke_H, smoke_active, smoke_original_out, smoke_rotated_out, smoke_opposite_out, smoke_loss, smoke_metrics, oracle_pair, zero_pair, oracle_cross, zero_cross
if DEVICE.type == "cuda": torch.cuda.empty_cache()


Pair oracle: 0.0000000 | all-zero baseline: 70.028
Cross-frame signed oracle: 0.0000000 | all-zero penalty: 61.571
Smoke objective: {'loss_total': 27.3758, 'loss_regular': 11.3339, 'loss_supervised': 11.4498, 'loss_active_supervised': 14.2485, 'loss_supervised_selection': 22.1362, 'loss_equivariance': 10.0219, 'loss_pair': 47.3854, 'loss_original': 12.7197, 'loss_rotated': 9.9482, 'weighted_active': 10.6864, 'weighted_equivariance': 0.5011, 'weighted_pair': 4.7385} gradient norm: 92.50337142984071
THREE-VIEW GRADIENT SMOKE: PASS


## 10. Optional short stochastic preflight (disabled by default for refinement)

Previous experiments completed stochastic testing; notebook 39b restarts from the 002.16 pretrained weights and keeps the smoke/oracle checks. The preflight below, if enabled, uses a **separate temporary optimizer** for a small number of steps and restores the initial weights. No stochastic pass/fail gate blocks the unattended full run. It reports supervised and signed-pair losses but should not be mistaken for proof of physical observability.


In [54]:
# Optional stochastic precheck; this experiment instead starts from 002.16 and has smoke/oracle tests.
RUN_STOCHASTIC_PRECHECK = True
STOCHASTIC_PRECHECK_STEPS = 500

if RUN_STOCHASTIC_PRECHECK:
    precheck_indices = np.linspace(0, len(train_dataset) - 1, min(BATCH_SIZE, 8), dtype=int)
    precheck_base = window_batch_to(collate_windows([train_dataset[int(i)] for i in precheck_indices]), DEVICE, dtype=MODEL_DTYPE)
    load_training_checkpoint(RESUME_CHECKPOINT, model=model, map_location=DEVICE, restore_rng_state=False)
    precheck_model_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    precheck_optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    precheck_generator = make_augmentation_generator(TRAIN_PAIR_SEED + 200000)
    for step in range(1, STOCHASTIC_PRECHECK_STEPS + 1):
        model.train()
        a, b, H, active, opposite = make_equivariant_windows(precheck_base, generator=precheck_generator, return_negative=True)
        precheck_optimizer.zero_grad(set_to_none=True)
        pa, pb = window_step(a).model_output, window_step(b).model_output
        pc = window_step(opposite).model_output if USE_SIGNED_PAIR else None
        objective, metrics = compute_equivariant_objective(a, pa, b, pb, H, active, opposite if USE_SIGNED_PAIR else None, pc)
        objective.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
        precheck_optimizer.step()
        if step == 1 or step % 20 == 0 or step == STOCHASTIC_PRECHECK_STEPS:
            print(f"Preflight {step}/{STOCHASTIC_PRECHECK_STEPS}: loss={metrics['loss_total']:.4f}, original={metrics['loss_original']:.4f}, rotated={metrics['loss_rotated']:.4f}, active={metrics['loss_active_supervised']:.4f}, pair={metrics['loss_pair']:.4f}", flush=True)
    model.load_state_dict(precheck_model_state)
    optimizer.zero_grad(set_to_none=True)
    del precheck_model_state, precheck_optimizer, precheck_generator, precheck_base
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
else:
    print("Stochastic precheck skipped. Previous 002.20 checkpoint is the training initialization; smoke/oracle checks ran above.")


KeyboardInterrupt: 

In [55]:
RUN_FULL_TRAINING = True  # Explicit unattended refinement; no inherited failing stochastic gate.
print("Full training enabled:", RUN_FULL_TRAINING)
print("Three forward passes per batch:", USE_SIGNED_PAIR, "(False -> two-view baseline)")
print("Pair regularizer weight:", PAIR_LOSS_WEIGHT, "; optional opposite supervised view weight:", OPPOSITE_VIEW_SUPERVISION_WEIGHT)


Full training enabled: True
Three forward passes per batch: True (False -> two-view baseline)
Pair regularizer weight: 0.1 ; optional opposite supervised view weight: 0.5


## 11. Single-window training/validation epoch runner (two or three forward passes)


In [56]:
def run_equivariant_epoch(loader, *, training: bool, generator: torch.Generator, global_step: int):
    model.train(training)
    metric_sums, num_samples = {}, 0
    start = time.perf_counter()
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch_number, base in enumerate(loader, start=1):
            base = window_batch_to(base, DEVICE, dtype=MODEL_DTYPE)
            if USE_SIGNED_PAIR:
                original, rotated, H, active, opposite = make_equivariant_windows(base, generator=generator, return_negative=True)
            else:
                original, rotated, H, active = make_equivariant_windows(base, generator=generator)
                opposite = None
            if training and (batch_number - 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.zero_grad(set_to_none=True)
            out_original = window_step(original).model_output
            out_rotated = window_step(rotated).model_output
            out_opposite = window_step(opposite).model_output if opposite is not None else None
            loss, metrics = compute_equivariant_objective(original, out_original, rotated, out_rotated, H, active, opposite, out_opposite)
            if training:
                group_index = (batch_number - 1) // GRADIENT_ACCUMULATION_STEPS
                group_size = min(GRADIENT_ACCUMULATION_STEPS, len(loader) - group_index * GRADIENT_ACCUMULATION_STEPS)
                (loss / group_size).backward()
                if batch_number % GRADIENT_ACCUMULATION_STEPS == 0 or batch_number == len(loader):
                    if GRADIENT_CLIP_NORM is not None:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
                    optimizer.step()
                    global_step += 1
            batch_size = next(iter(original.streams.values())).values.shape[0]
            accumulate_metrics(metric_sums, metrics, batch_size)
            num_samples += batch_size
            if training and PRINT_EVERY_BATCHES > 0 and batch_number % PRINT_EVERY_BATCHES == 0:
                print(f"batch {batch_number}/{len(loader)} total={metrics['loss_total']:.5f} sup={metrics['loss_supervised']:.5f} active={metrics['loss_active_supervised']:.5f} equiv={metrics['loss_equivariance']:.5f} pair={metrics['loss_pair']:.5f}", flush=True)
    elapsed = time.perf_counter() - start
    epoch_metrics = normalize_metrics(metric_sums, num_samples)
    epoch_metrics["epoch_time_s"] = elapsed
    epoch_metrics["samples_per_s"] = num_samples / max(elapsed, 1e-9)
    return epoch_metrics, global_step


def accumulate_metrics(metric_sums, metrics, batch_size):
    for key, value in metrics.items():
        metric_sums[key] = metric_sums.get(key, 0.0) + float(value) * batch_size


def normalize_metrics(metric_sums, num_samples):
    if num_samples <= 0:
        raise ValueError("No examples in epoch.")
    return {key: value / num_samples for key, value in metric_sums.items()}


## 12. Experiment and calibration checkpoint metadata


In [57]:
RUN_DIR = ROOT / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT_PATH = RUN_DIR / "best.pt"
LAST_CHECKPOINT_PATH = RUN_DIR / "last.pt"
METRICS_PATH = RUN_DIR / "metrics.jsonl"
EXPERIMENT_SNAPSHOT = {
    "run_name": RUN_NAME, "architecture": "PooledCalibrationModel", "model_config": model_config,
    "microbatch_size": BATCH_SIZE, "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "windowing_config": windowing_config, "rig_calibration": rig_calibration, "measurement_dim": MEASUREMENT_DIM,
    "calibration_keys": CALIBRATION_KEYS,
    "sequence_split": {"train": tuple(TRAIN_NAMES), "validation": tuple(VAL_NAMES), "test": tuple(TEST_NAMES)},
    "paired_prior_training": USE_SIGNED_PAIR,
    "pair_sampling_strategy": "single_truncated_student_t_one_calibration_per_sample_with_optional_opposite_prior",
    "pair_prior_config": pair_prior_config, "context_augmentation_config": context_augmentation_config,
    "equivariance_loss_weight": EQUIVARIANCE_LOSS_WEIGHT, "paired_view_probability": COMMON_BODY_FRAME_PROBABILITY,
    "active_rotation_supervision_weight": ACTIVE_ROTATION_SUPERVISION_WEIGHT, "rotated_active_multiplier": ROTATED_ACTIVE_MULTIPLIER, "signed_pair_enabled": USE_SIGNED_PAIR, "signed_pair_weight": PAIR_LOSS_WEIGHT, "signed_pair_variant": "cross_frame_rotated_vs_transformed_opposite_rotation", "opposite_view_supervision_weight": OPPOSITE_VIEW_SUPERVISION_WEIGHT,
    "paired_view_transform": "shared_left_rotation_only_SO3",
    "training_objective": "supervised_views_plus_rotated_active_rotation_plus_equivariance_plus_cross_frame_signed_rotation",
    "equivariance_components": "rotation_and_translation_adjoint_plus_tau_identity",
    "pair_rotation_max_rad": PAIR_ROTATION_MAX_RAD, "pair_time_offset_max_s": PAIR_TIME_OFFSET_MAX_S,
    "initialization_checkpoint": str(RESUME_CHECKPOINT), "checkpoint_selection_metric": "loss_supervised_selection",
    "true_calibration_rotation_max_rad": TRUE_CALIBRATION_ROTATION_MAX_RAD,
    "true_calibration_time_offset_max_s": TRUE_CALIBRATION_TIME_OFFSET_MAX_S,
    "seeds": {"random": RANDOM_SEED, "train": TRAIN_PAIR_SEED, "validation": VALIDATION_PAIR_SEED, "test": TEST_PAIR_SEED},
}
LOGGER_EXPERIMENT_SNAPSHOT = {
    "run_name": RUN_NAME, "architecture": "PooledCalibrationModel", "calibration_keys": list(CALIBRATION_KEYS),
    "training_objective": EXPERIMENT_SNAPSHOT["training_objective"], "equivariance_loss_weight": EQUIVARIANCE_LOSS_WEIGHT,
    "paired_view_probability": COMMON_BODY_FRAME_PROBABILITY, "prior_rotation_max_deg": float(np.rad2deg(PAIR_ROTATION_MAX_RAD)),
    "true_calibration_rotation_max_deg": float(np.rad2deg(TRUE_CALIBRATION_ROTATION_MAX_RAD)),
    "active_rotation_supervision_weight": ACTIVE_ROTATION_SUPERVISION_WEIGHT, "rotated_active_multiplier": ROTATED_ACTIVE_MULTIPLIER, "signed_pair_enabled": USE_SIGNED_PAIR, "signed_pair_weight": PAIR_LOSS_WEIGHT, "signed_pair_variant": "cross_frame_rotation",
    "source_checkpoint": str(RESUME_CHECKPOINT), "checkpoint_selection_metric": "loss_supervised_selection",
}
train_equiv_generator = make_augmentation_generator(TRAIN_PAIR_SEED)


## 13. Load 002.16 calibration-only model weights (fresh optimizer and scheduler)


In [58]:
start_epoch = 0
global_step = 0
best_validation_loss = float("inf")

if RESUME_CHECKPOINT is not None:
    checkpoint = load_training_checkpoint(
        RESUME_CHECKPOINT,
        model=model,
        # Initialize model weights only. The optimizer, scheduler and RNG state are fresh.
        # optimizer=optimizer,
        # scheduler=scheduler,
        map_location=DEVICE,
        restore_rng_state=False,
    )
    # start_epoch = int(checkpoint.get("epoch", 0))
    # global_step = int(checkpoint.get("global_step", 0))
    # best_validation_loss = float(checkpoint.get("extra_state", {}).get("best_validation_loss", float("inf")))

    print("Initialized model weights from:", RESUME_CHECKPOINT)
    print("start epoch:", start_epoch)
    print("global step:", global_step)


Initialized model weights from: /home/camel/Skoltech/Mobile_Robotics_Lab/nn-obs-calib/runs/kaist_pooled_calibration_paired_002.16_correction_only/best.pt
start epoch: 0
global step: 0


## 14. Full 120-epoch signed cross-frame training from 002.16

The default runs without a stochastic gating override. A new run directory prevents overwriting 002.16, 002.20, or 002.21; checkpoints store exact model/optimizer/scheduler state and the experiment config. The best checkpoint is selected by supervised original/rotated/opposite performance with active-rotation emphasis, not by pair/equivariance consistency alone. Cosine decay progresses every epoch, whether or not validation improves.


In [59]:
if DEVICE.type == "cuda": torch.cuda.empty_cache()


In [ ]:
history = []
print("RUN_FULL_TRAINING:", RUN_FULL_TRAINING, "| source:", RESUME_CHECKPOINT)
if RUN_FULL_TRAINING:
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    with MetricsLogger(METRICS_PATH, append=False) as logger:
        logger.log_config(model_config, name="model_config")
        logger.log_config(context_augmentation_config, name="context_augmentation_config")
        logger.log_config(pair_prior_config, name="prior_config")
        logger.log_config(LOGGER_EXPERIMENT_SNAPSHOT, name="experiment")
        for epoch in range(start_epoch, NUM_EPOCHS):
            train_metrics, global_step = run_equivariant_epoch(train_loader, training=True, generator=train_equiv_generator, global_step=global_step)
            val_generator = make_augmentation_generator(VALIDATION_PAIR_SEED)
            val_metrics, _ = run_equivariant_epoch(val_loader, training=False, generator=val_generator, global_step=global_step)
            selection_loss = val_metrics["loss_supervised_selection"]
            lr_used = optimizer.param_groups[0]["lr"]
            scheduler.step()
            row = {"epoch": epoch + 1, "global_step": global_step, "learning_rate": lr_used, **{f"train/{k}": v for k, v in train_metrics.items()}, **{f"val/{k}": v for k, v in val_metrics.items()}}
            history.append(row)
            logger.log_metrics({**train_metrics, "learning_rate": lr_used}, split="train", epoch=epoch + 1, step=global_step)
            logger.log_metrics(val_metrics, split="validation", epoch=epoch + 1, step=global_step)
            print(f"Epoch {epoch+1}/{NUM_EPOCHS}: train supervised={train_metrics['loss_supervised']:.4f} active={train_metrics['loss_active_supervised']:.4f} equiv={train_metrics['loss_equivariance']:.4f} pair={train_metrics['loss_pair']:.4f} | val supervised={val_metrics['loss_supervised']:.4f} (orig={val_metrics['loss_original']:.4f}, rotated={val_metrics['loss_rotated']:.4f}) active={val_metrics['loss_active_supervised']:.4f} pair={val_metrics['loss_pair']:.4f} | val activeRot orig={val_metrics['original/active_rotation_error_deg']:.2f}deg rotated={val_metrics['rotated/active_rotation_error_deg']:.2f}deg | gapPred={val_metrics.get('pair/predicted_rotation_gap_deg', 0.0):.1f}deg/gapTrue={val_metrics.get('pair/target_rotation_gap_deg', 0.0):.1f}deg | activeMagPredRot={val_metrics.get('rotated/active_rotation_prediction_deg', 0.0):.1f}deg/target={val_metrics.get('rotated/active_rotation_target_deg', 0.0):.1f}deg | select={selection_loss:.4f} lr={lr_used:.2e}", flush=True)
            if selection_loss < best_validation_loss:
                best_validation_loss = selection_loss
                save_training_checkpoint(BEST_CHECKPOINT_PATH, model=model, optimizer=optimizer, scheduler=scheduler, epoch=epoch + 1, global_step=global_step, metrics=val_metrics, generators={"train_shuffle": train_shuffle_generator, "train_pair": train_equiv_generator}, extra_state={"best_validation_loss": best_validation_loss, "experiment": EXPERIMENT_SNAPSHOT})
                print("Saved best checkpoint.")
            if (epoch + 1) % CHECKPOINT_EVERY_EPOCHS == 0 or epoch + 1 == NUM_EPOCHS:
                save_training_checkpoint(LAST_CHECKPOINT_PATH, model=model, optimizer=optimizer, scheduler=scheduler, epoch=epoch + 1, global_step=global_step, metrics=val_metrics, generators={"train_shuffle": train_shuffle_generator, "train_pair": train_equiv_generator}, extra_state={"best_validation_loss": best_validation_loss, "experiment": EXPERIMENT_SNAPSHOT})
            gc.collect()
            if DEVICE.type == "cuda": torch.cuda.empty_cache()
else:
    print("Training disabled in configuration. The source checkpoint and smoke checks are unaffected.")


RUN_FULL_TRAINING: True | source: /home/camel/Skoltech/Mobile_Robotics_Lab/nn-obs-calib/runs/kaist_pooled_calibration_paired_002.16_correction_only/best.pt
Epoch 1/200: train supervised=6.9410 active=13.5972 equiv=15.6042 pair=30.0345 | val supervised=5.1544 (orig=0.7966, rotated=11.6556) active=10.8804 pair=22.2136 | val activeRot orig=6.09deg rotated=20.33deg | gapPred=23.0deg/gapTrue=41.4deg | activeMagPredRot=3.7deg/target=20.7deg | select=13.3146 lr=1.00e-05
Saved best checkpoint.
Epoch 2/200: train supervised=5.4503 active=11.6749 equiv=12.0241 pair=25.0962 | val supervised=5.0327 (orig=0.7520, rotated=11.4093) active=10.7098 pair=20.8120 | val activeRot orig=6.11deg rotated=20.15deg | gapPred=23.6deg/gapTrue=41.4deg | activeMagPredRot=3.6deg/target=20.7deg | select=13.0650 lr=1.00e-05
Saved best checkpoint.
Epoch 3/200: train supervised=5.3854 active=11.6509 equiv=12.1960 pair=24.1812 | val supervised=4.9634 (orig=0.7245, rotated=11.2966) active=10.6473 pair=20.0096 | val active

## 15. Checkpoint snapshot verification

In [ ]:
from obscalib.training.checkpoint import read_training_checkpoint
for checkpoint_path in (BEST_CHECKPOINT_PATH, LAST_CHECKPOINT_PATH):
    if not checkpoint_path.is_file():
        print("Not yet created:", checkpoint_path)
        continue
    saved = read_training_checkpoint(checkpoint_path, map_location="cpu")
    snapshot = (saved.get("extra_state") or {}).get("experiment", {})
    assert snapshot.get("run_name") == RUN_NAME
    assert snapshot.get("training_objective") == EXPERIMENT_SNAPSHOT["training_objective"]
    assert isinstance(snapshot.get("model_config"), PooledCalibrationModelConfig)
    assert isinstance(snapshot.get("windowing_config"), WindowingConfig)
    print("Verified:", checkpoint_path.name, "epoch:", saved.get("epoch"))


## 16. Validation curves: supervised, active-rotation, equivariance and optional pair


In [ ]:
if history:
    epochs = [r["epoch"] for r in history]
    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
    for phase in ("train", "val"):
        for key in ("loss_original", "loss_rotated", "loss_supervised_selection"):
            axes[0].plot(epochs, [r[f"{phase}/{key}"] for r in history], label=f"{phase} {key}")
        for key in ("loss_equivariance", "loss_pair", "loss_active_supervised"):
            axes[1].plot(epochs, [r[f"{phase}/{key}"] for r in history], label=f"{phase} {key}")
    for view in ("original", "rotated"):
        axes[2].plot(epochs, [r[f"val/{view}/active_rotation_error_deg"] for r in history], label=f"validation {view} active rotation")
    axes[0].set_ylabel("Supervised loss")
    axes[1].set_ylabel("Auxiliary raw loss")
    axes[2].set_ylabel("Active correction error [deg]")
    axes[2].set_xlabel("Epoch")
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
    plt.figure(figsize=(10, 3))
    plt.plot(epochs, [r["learning_rate"] for r in history])
    plt.title("Cosine learning-rate refinement")
    plt.grid(alpha=0.25)
    plt.show()


## 17. Held-out evaluation of the best checkpoint and source model

In [ ]:
COMPARE_TO_SOURCE_CHECKPOINT = True

def evaluate_matched_test():
    gen = make_augmentation_generator(TEST_PAIR_SEED + 901)
    result, _ = run_equivariant_epoch(test_loader, training=False, generator=gen, global_step=0)
    return result


def print_evaluation(label, m):
    print(f"{label}: total={m['loss_total']:.4f}, regular={m['loss_regular']:.4f}, original={m['loss_original']:.4f}, rotated={m['loss_rotated']:.4f}, equiv={m['loss_equivariance']:.4f}, pair={m['loss_pair']:.4f}, active={m['loss_active_supervised']:.4f}")
    for key in CALIBRATION_KEYS:
        print(f"   {key:12s} original rot={m[f'original/{key}/rotation_error_deg']:.2f}deg, rotated rot={m[f'rotated/{key}/rotation_error_deg']:.2f}deg, equiv={m[f'equiv/{key}/rotation_equivariance_deg']:.2f}deg, original tau={m[f'original/{key}/time_offset_error_ms']:.2f}ms, rotated tau={m[f'rotated/{key}/time_offset_error_ms']:.2f}ms")

if BEST_CHECKPOINT_PATH.is_file():
    load_training_checkpoint(BEST_CHECKPOINT_PATH, model=model, map_location=DEVICE, restore_rng_state=False)
    new_metrics = evaluate_matched_test()
    print_evaluation("New checkpoint",new_metrics)
    if COMPARE_TO_SOURCE_CHECKPOINT:
        load_training_checkpoint(RESUME_CHECKPOINT, model=model, map_location=DEVICE, restore_rng_state=False)
        try:
            source_metrics = evaluate_matched_test()
            print_evaluation("Pretrained source", source_metrics)
            print("Change in rotated supervised loss: %+.1f%%" % (100*(new_metrics["loss_rotated"]/source_metrics["loss_rotated"] -1)))
            print("Change in equivariance loss: %+.1f%%" % (100*(new_metrics["loss_equivariance"]/max(source_metrics["loss_equivariance"],1e-10) -1)))
        finally:
            load_training_checkpoint(BEST_CHECKPOINT_PATH, model=model, map_location=DEVICE, restore_rng_state=False)
else:
    print("Best checkpoint absent. Train first or inspect the fixed stochastic diagnostic.")


## 18. Paired-view correction and equivariance scatter

The two views have the same physical target. The rotated-view predicted correction is brought back to the original common frame by \(G^\top\) for comparison. Plots show both frame-wise supervised errors **and** differences after compensating for \(G\).

In [ ]:
VIS_SEQUENCE_NAME = "Urban38"
VIS_START_WINDOW = 0
VIS_NUM_WINDOWS = 300
VIS_BATCH_SIZE = 16
sequence_indices = [i for i in range(len(test_dataset)) if test_dataset.sequence_name_for_window(i) == VIS_SEQUENCE_NAME]
selected_indices = sequence_indices[VIS_START_WINDOW:VIS_START_WINDOW+VIS_NUM_WINDOWS]
if not selected_indices: raise ValueError("No selected held-out windows.")


def collect_equivariant_predictions(dataset, indices, batch_size=VIS_BATCH_SIZE, seed=VISUALIZATION_PAIR_SEED):
    model.eval()
    generator=make_augmentation_generator(seed)
    rows=[]
    with torch.no_grad():
        for start in range(0,len(indices),batch_size):
            batch=window_batch_to(collate_windows([dataset[int(j)] for j in indices[start:start+batch_size]]),DEVICE,dtype=MODEL_DTYPE)
            a,b,H,active=make_equivariant_windows(batch,generator=generator,frame_probability=1.0)
            pa,pb=window_step(a).model_output,window_step(b).model_output
            G=H[:,:3,:3]
            for key in CALIBRATION_KEYS:
                target_xi,target_tau=correction_target(a,key)
                xa=pa.predictions[key].delta_xi
                xb=pb.predictions[key].delta_xi
                xa_in_b=rotate_twist(xa,G)
                xb_in_a=rotate_twist(xb,G.transpose(1,2))
                for i in range(xa.shape[0]):
                    rows.append({"window":int(indices[start+i]),"sensor":key,"target_rotation_deg":float(torch.rad2deg(torch.linalg.vector_norm(target_xi[i,:3])).cpu()),"original_predicted_rotation_deg":float(torch.rad2deg(torch.linalg.vector_norm(xa[i,:3])).cpu()),"rotated_predicted_rotation_deg":float(torch.rad2deg(torch.linalg.vector_norm(xb[i,:3])).cpu()),"original_error_deg":float(torch.rad2deg(torch.linalg.vector_norm(xa[i,:3]-target_xi[i,:3])).cpu()),"rotated_error_deg":float(torch.rad2deg(torch.linalg.vector_norm(xb_in_a[i,:3]-target_xi[i,:3])).cpu()),"equivariance_error_deg":float(torch.rad2deg(torch.linalg.vector_norm(xb[i,:3]-xa_in_b[i,:3])).cpu()),"original_tau_error_ms":float((1000*(pa.predictions[key].delta_tau[i]-target_tau[i]).abs()).cpu()),"rotated_tau_error_ms":float((1000*(pb.predictions[key].delta_tau[i]-target_tau[i]).abs()).cpu()),"tau_equivariance_ms":float((1000*(pb.predictions[key].delta_tau[i]-pa.predictions[key].delta_tau[i]).abs()).cpu())})
    return __import__('pandas').DataFrame(rows)

if BEST_CHECKPOINT_PATH.is_file():
    load_training_checkpoint(BEST_CHECKPOINT_PATH,model=model,map_location=DEVICE,restore_rng_state=False)
    vis_equiv = collect_equivariant_predictions(test_dataset, selected_indices)
    vis_equiv.to_csv(RUN_DIR/"heldout_frame_equivariance.csv",index=False)
    display(vis_equiv.groupby("sensor")[["original_error_deg","rotated_error_deg","equivariance_error_deg","original_tau_error_ms","rotated_tau_error_ms","tau_equivariance_ms"]].agg(["mean","median"]).round(3))
    fig,axes=plt.subplots(1,3,figsize=(17,4.5))
    for key,group in vis_equiv.groupby("sensor"):
        axes[0].scatter(group.target_rotation_deg,group.original_error_deg,alpha=.25,label=key)
        axes[1].scatter(group.target_rotation_deg,group.rotated_error_deg,alpha=.25,label=key)
        axes[2].scatter(group.target_rotation_deg,group.equivariance_error_deg,alpha=.25,label=key)
    for ax,title in zip(axes,("Original-frame supervised error","Rotated-frame supervised error","Equivariance discrepancy")):
        ax.set(xlabel="Target rotation [deg]",ylabel="Error [deg]",title=title);ax.grid(alpha=.2)
    axes[0].legend(fontsize=8);fig.tight_layout();plt.show()
else:
    print("Visualization skipped until a trained checkpoint is available.")


## 18b. Diagnosing correction shrinkage, not just low median error

For `one_calibration_per_sample=True`, roughly two of three sensor targets are zero. **Never interpret the overall median as calibration accuracy on an active fault.** On active faults plot predicted rotation *magnitude*, correction *error*, and the zero-prediction baseline. A true zero-output model has predicted magnitude near zero and error equal to the target magnitude.

The ratio `predicted magnitude / target magnitude` can exceed one even when the direction is wrong, so combine it with the actual error and improvement relative to the zero predictor. Aggregate by target-angle bins on the held-out set. These are Euclidean rotation-vector errors, matching the supervised objective; they are not geodesic post-update trajectory errors.

In [ ]:

import pandas as pd

if "vis_equiv" not in globals():
    raise RuntimeError("Run held-out evaluation / section 18 before inspecting active corrections.")
active_vis = vis_equiv[vis_equiv["target_rotation_deg"] >= 1.0].copy()
if active_vis.empty:
    raise RuntimeError("No actively perturbed rotation targets in visualization sample.")
active_vis["rotation_bin"] = pd.cut(active_vis["target_rotation_deg"], bins=[1, 5, 15, 30, 45, 60, 180], include_lowest=True)
for view in ("original", "rotated"):
    active_vis[f"{view}_magnitude_ratio"] = active_vis[f"{view}_predicted_rotation_deg"] / active_vis["target_rotation_deg"].clip(lower=1.0)
    active_vis[f"{view}_gain_over_zero_deg"] = active_vis["target_rotation_deg"] - active_vis[f"{view}_error_deg"]

summary = active_vis.groupby(["sensor", "rotation_bin"], observed=True).agg(
    count=("target_rotation_deg", "size"), target_deg=("target_rotation_deg", "median"),
    original_error_deg=("original_error_deg", "mean"), rotated_error_deg=("rotated_error_deg", "mean"),
    original_gain_over_zero_deg=("original_gain_over_zero_deg", "mean"), rotated_gain_over_zero_deg=("rotated_gain_over_zero_deg", "mean"),
    original_predicted_to_target=("original_magnitude_ratio", "median"), rotated_predicted_to_target=("rotated_magnitude_ratio", "median"),
).reset_index()
display(summary.round(3))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for key, group in active_vis.groupby("sensor"):
    axes[0].scatter(group["target_rotation_deg"], group["rotated_predicted_rotation_deg"], alpha=0.3, label=key)
    axes[1].scatter(group["target_rotation_deg"], group["rotated_error_deg"], alpha=0.3, label=key)
    axes[2].scatter(group["target_rotation_deg"], group["rotated_gain_over_zero_deg"], alpha=0.3, label=key)
max_angle = max(60.0, float(active_vis["target_rotation_deg"].max()))
axes[0].plot([0, max_angle], [0, max_angle], "k--", label="Ideal magnitude")
axes[1].plot([0, max_angle], [0, max_angle], "k--", label="Zero-prediction baseline")
axes[2].axhline(0, color="black", linestyle="--", label="Zero-prediction baseline")
axes[0].set(title="Rotated prediction magnitude", ylabel="Predicted magnitude [deg]")
axes[1].set(title="Rotated correction error", ylabel="Prediction error [deg]")
axes[2].set(title="Gain over a zero prediction", ylabel="Gain [deg]")
for ax in axes:
    ax.set_xlabel("Active target magnitude [deg]")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()
active_vis.to_csv(RUN_DIR / "heldout_active_correction_shrinkage.csv", index=False)
print("Mean rotated active correction gain versus zero [deg]:", active_vis["rotated_gain_over_zero_deg"].mean())


## 19. Exact-truth fixed-point diagnostic

A model can reduce equivariance loss by predicting zero for both views. Therefore retain the pretrained fixed-point test: nominally correct calibration should receive nearly zero corrections. This is an extra check, not a substitute for perturbed-calibration accuracy.

In [ ]:
FIXED_POINT_NUM_WINDOWS = min(100, len(sequence_indices))
fixed_point_positions = np.unique(np.linspace(0, len(sequence_indices) - 1, FIXED_POINT_NUM_WINDOWS).round().astype(int))
fixed_point_indices = np.asarray(sequence_indices)[fixed_point_positions]

fixed_point_results = {calibration_key: {"delta_xi": [], "delta_tau": []} for calibration_key in CALIBRATION_KEYS}

model.eval()

with torch.no_grad():
    for dataset_index in fixed_point_indices:
        window = window_batch_to(collate_windows([test_dataset[int(dataset_index)]]), DEVICE, dtype=MODEL_DTYPE)
        exact_true_calibration = {calibration_key: CalibrationState(transform=state.transform.clone(), time_offset=state.time_offset.clone()) for calibration_key, state in window.current_calibration.items()}
        result = window_step(window, calibration=exact_true_calibration)

        for calibration_key, prediction in result.model_output.predictions.items():
            fixed_point_results[calibration_key]["delta_xi"].append(prediction.delta_xi[0].detach().cpu())
            fixed_point_results[calibration_key]["delta_tau"].append(prediction.delta_tau[0, 0].detach().cpu())

for calibration_key in CALIBRATION_KEYS:
    fixed_point_results[calibration_key]["delta_xi"] = torch.stack(fixed_point_results[calibration_key]["delta_xi"]).numpy()
    fixed_point_results[calibration_key]["delta_tau"] = torch.stack(fixed_point_results[calibration_key]["delta_tau"]).numpy()

component_names = ("phi_x", "phi_y", "phi_z", "rho_x", "rho_y", "rho_z", "tau")

print("=" * 120)
print("EXACT-TRUTH FIXED-POINT DIAGNOSTIC")
print("=" * 120)

for calibration_key in CALIBRATION_KEYS:
    result = fixed_point_results[calibration_key]
    prediction = np.concatenate([result["delta_xi"], result["delta_tau"][:, None]], axis=1)

    print(calibration_key)
    print(f"{'component':8s} {'mean':>16s} {'mean |x|':>16s} {'RMSE':>16s} {'std':>16s} {'range':>16s}")

    for component_index, component_name in enumerate(component_names):
        values = prediction[:, component_index]

        if component_index < 3:
            values, unit = np.rad2deg(values), "deg"
        elif component_index < 6:
            values, unit = 100.0 * values, "cm"
        else:
            values, unit = 1000.0 * values, "ms"

        mean = np.mean(values)
        mean_abs = np.mean(np.abs(values))
        rmse = np.sqrt(np.mean(values ** 2))
        std = np.std(values)
        value_range = np.max(values) - np.min(values)

        print(f"{component_name:8s} {mean:+12.6f} {unit:3s} {mean_abs:12.6f} {unit:3s} {rmse:12.6f} {unit:3s} {std:12.6f} {unit:3s} {value_range:12.6f} {unit:3s}")

    print()


## 20. Interpretation and next evaluation

- **Active rotated correction gain over zero > 0**: the model responds to the perturbation rather than just outputting zero.
- **Pair gap increases correctly but active error does not improve**: signed pair supervision is still not enough to learn the physically correct correction.
- **Equivariance falls but active correction remains weak**: view consistency alone is not meaningful calibration success.
- **Original-view performance degrades**: lower auxiliary weights or prefer the previous 002.20 checkpoint.

After the unattended run, compare `best.pt` against 002.20 on notebook 37's held-out NewCollege diagnostic before concluding anything about cross-domain transfer. The 2.5M-parameter Transformer receives no additional physical information from opposite sign priors: pair loss is an optimization constraint, not a new sensor or source of observability.


In [ ]:
print("Run:", RUN_NAME)
print("Source checkpoint:", RESUME_CHECKPOINT)
print("Regime: independent 5-second windows, frame equivariance and optional opposite-sign prior (NO BPTT)")
print("Rotated-view probability:", COMMON_BODY_FRAME_PROBABILITY)
print("Equivariance weight:", EQUIVARIANCE_LOSS_WEIGHT)
print("Signed pair enabled:", USE_SIGNED_PAIR)
print("Best checkpoint:", BEST_CHECKPOINT_PATH, "exists:", BEST_CHECKPOINT_PATH.is_file())
if history:
    best_row=min(history,key=lambda r:r['val/loss_supervised_selection'])
    print("Best validation epoch:",best_row['epoch'], "regular:",best_row['val/loss_supervised_selection'], "equiv:",best_row['val/loss_equivariance'])
